# MCP 第 06 课：Function Calling 深入，模型到底在“调用”什么

到这里你已经知道 MCP 和 tool use 的基本关系了。

这一课我们聚焦一个更细的点：

> 当我们说模型在“function calling”时，真正发生的到底是什么？

最重要的答案是：

**模型并没有直接执行函数，它只是生成了一个结构化的“调用意图”。**


## 1. Function Calling 的三层

你可以把它拆成三层：

- 模型层：根据上下文判断要不要调用工具
- 协议层：把调用意图输出成结构化数据
- 执行层：你的程序真正执行函数、命令或 API

很多初学者把这三层混在一起，所以会误以为“模型自己能执行工具”。


## 2. 一个最小 schema

绝大多数 function calling 都离不开这几部分：

- `name`
- `description`
- `parameters` / JSON Schema

schema 的任务不是“给人看”，而是**约束模型输出的参数形状**。


In [ ]:
weather_tool = {
    "type": "function",
    "function": {
        "name": "get_weather",
        "description": "Get current weather for a city.",
        "parameters": {
            "type": "object",
            "properties": {
                "city": {"type": "string"},
                "unit": {"type": "string", "enum": ["celsius", "fahrenheit"]},
            },
            "required": ["city"],
        },
    },
}

weather_tool


## 3. `tool_choice` 的含义

这是非常关键但经常被忽略的控制项。

常见模式：

- `auto`：模型自己决定要不要调工具
- `required`：必须至少调一个工具
- 指定某个工具：强制调用某个具体函数

它本质上是在控制“模型的决策自由度”。


In [ ]:
examples = {
    "auto": "让模型自己判断是否需要工具",
    "required": "要求模型必须走工具链路",
    "specific_tool": "例如强制使用 get_weather",
}

examples


## 4. 为什么参数约束很重要

真实工程里，参数 schema 至少解决四件事：

- 减少参数名漂移
- 限制枚举值
- 明确必填字段
- 为执行层做前置验证

schema 写得越松，执行层越痛苦。


In [ ]:
strict_tool = {
    "type": "object",
    "properties": {
        "path": {"type": "string"},
        "mode": {"type": "string", "enum": ["read", "write"]},
    },
    "required": ["path", "mode"],
    "additionalProperties": False,
}

strict_tool


## 5. 并行 tool calls 和串行 tool calls

有些任务天然适合并行：

- 同时查多个城市天气
- 同时读多个文档片段

有些任务必须串行：

- 先搜索文件，再决定改哪个文件
- 先跑测试，再根据失败结果修代码

**是否并行，取决于数据依赖，而不是取决于你有没有多个工具。**


## 6. 失败、重试与幂等性

真正难的地方不在“第一次调用成功”，而在失败之后怎么办。

你至少要问：

- 这个工具能安全重试吗？
- 它有副作用吗？
- 失败时要把什么信息回给模型？

读型工具通常更容易重试，写型工具要更谨慎。


## 7. 和 MCP 的关系

Function calling 更像“模型怎么表达想调用工具”。

MCP 更像“应用和外部能力之间怎么标准化通信”。

两者不是替代关系，常常是上下两层：

- 上层 LLM 用 function calling 做决策
- 下层执行用 MCP 去发现和调用实际能力


## 本课工程直觉

1. **Function calling 的本质是“结构化意图输出”，不是模型直接执行。**
2. **schema 既是提示词的一部分，也是执行层的防线。**
3. **复杂工具链路真正的难点在重试、回填、幂等和权限。**
